In [ ]:
%config InlineBackend.figure_format = "retina"
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

sns.set_context("talk")

In [ ]:
import pickle

all_models = {
    "OmniRad-Small": "omni-outputs.pkl",
    "OmniRad-Small (WeightedCE)": "omniWCE-outputs.pkl",
    "ViT": "vit-outputs.pkl",
    "ViT (WeightedCE)": "vitWCE-outputs.pkl",
}

for model_name in all_models:
    with open(os.path.join("outputs", all_models[model_name]), "rb") as f:
        outputs = pickle.load(f)
        all_models[model_name] = outputs
    

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np

def to_numpy(x):
    return x.cpu().numpy() if hasattr(x, "cpu") else np.asarray(x)

plt.figure(figsize=(8, 6))
palette = sns.color_palette("tab10", n_colors=len(all_models))

for (model_name, outputs), color in zip(all_models.items(), palette):
    probs = to_numpy(outputs["probabilities"])[:, 1]
    labels = to_numpy(outputs["labels"])
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    sns.lineplot(x=fpr, y=tpr, label=f"{model_name}", color=color, errorbar=None)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — All Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(8, 6))
palette = sns.color_palette("tab10", n_colors=len(all_models))

for (model_name, outputs), color in zip(all_models.items(), palette):
    probs = to_numpy(outputs["probabilities"])[:, 1]
    labels = to_numpy(outputs["labels"])
    precision, recall, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    sns.lineplot(x=recall, y=precision, label=f"{model_name}", color=color, errorbar=None)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves — All Models")
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
# Check labels are all equal, if they are (they should be) they've all be run in the same order
# which makes merging into a single dataframe easier
all(np.array_equal(to_numpy(outputs["labels"]), to_numpy(outputs["labels"])) for outputs in all_models.values())

In [ ]:
# Plot decision net benefit curves
# First of all we need to get the data in the format that dcurves wants
import pandas as pd

dca_df = pd.DataFrame()

# We need a dataframe with columns for each model's predicted probabilities and a column for the true labels
for i in range(len(all_models["ViT"]["labels"])):
    row = {"label": all_models["ViT"]["labels"][i].item()}
    for model_name, outputs in all_models.items():
        row[model_name] = outputs["probabilities"][i][1].item()
    dca_df = pd.concat([dca_df, pd.DataFrame([row])], ignore_index=True)
    

In [ ]:
# Now we can go decision curve analysis using the dcurves package
import dcurves

dca_results_df = dcurves.dca(
    data=dca_df,
    outcome="label",
    modelnames=list(all_models.keys()),
)

In [ ]:
# Seaborn decision curve plot with manual legend control

plot_df = dca_results_df.copy()

model_order = list(all_models.keys())
for baseline in ["all", "none"]:
    if baseline in plot_df["model"].unique():
        model_order.append(baseline)

palette = {name: color for name, color in zip(all_models.keys(), sns.color_palette("tab10", n_colors=len(all_models)))}
if "all" in plot_df["model"].unique():
    palette["all"] = "black"
if "none" in plot_df["model"].unique():
    palette["none"] = "gray"

fig, ax = plt.subplots(figsize=(8, 6))

sns.lineplot(
    data=plot_df,
    x="threshold",
    y="net_benefit",
    hue="model",
    hue_order=model_order,
    palette=palette,
    linewidth=2,
    ax=ax,
)

ax.set_xlim(0, 0.9)
ax.set_ylim(-0.05, 0.25)
ax.set_xlabel("Threshold probability")
ax.set_ylabel("Net benefit")
ax.set_title("Clinical Net Benefit Analysis")
ax.legend(title="", loc="upper right", prop={"size": 11})
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for each model, computed from the saved outputs
id2label = {0: "Non-endometriosis", 1: "Endometriosis"}

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

for ax, (model_name, outputs) in zip(axes.flat, all_models.items()):
    cm = confusion_matrix(to_numpy(outputs["labels"]), to_numpy(outputs["predictions"]))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    annot = [[f"{count}\n({pct:.1%})" for count, pct in zip(row_cm, row_norm)] for row_cm, row_norm in zip(cm, cm_norm)]

    sns.heatmap(
        cm_norm,
        annot=annot,
        fmt="",
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=id2label.values(),
        yticklabels=id2label.values(),
        ax=ax,
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(model_name)

fig.suptitle("Endometriosis Confusion Matrices")
plt.tight_layout()
plt.show()